# Experiment 17: Mobile Price Prediction Using Machine Learning

**Aim:** To predict the price range of mobile phones using a Random Forest classification algorithm.

**Dataset note:** This notebook creates a reproducible sample mobile-phone dataset in the notebook itself. It uses specifications such as battery power, RAM, internal memory, camera and screen size. The target has four price categories (0–3). Replace the sample data with a real mobile-price CSV when available.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Create a reproducible sample mobile-phone dataset
rng = np.random.default_rng(42)
n = 800

mobile = pd.DataFrame({
    "battery_power": rng.integers(500, 2001, n),
    "ram": rng.integers(256, 8001, n),
    "internal_memory": rng.choice([8, 16, 32, 64, 128, 256], n),
    "primary_camera": rng.integers(2, 49, n),
    "front_camera": rng.integers(0, 21, n),
    "screen_size": rng.uniform(4.5, 7.0, n).round(2),
    "mobile_weight": rng.integers(100, 251, n)
})

# Generate price categories from phone specifications
score = (
    mobile["ram"] / 8000 * 0.55
    + mobile["battery_power"] / 2000 * 0.20
    + mobile["internal_memory"] / 256 * 0.15
    + mobile["primary_camera"] / 48 * 0.10
)
mobile["price_range"] = pd.qcut(
    score.rank(method="first"), 4, labels=[0, 1, 2, 3]
).astype(int)

print("Sample mobile dataset:")
display(mobile.head())
print("Number of records:", len(mobile))
print("\nPrice-range distribution:")
print(mobile["price_range"].value_counts().sort_index())

Sample mobile dataset:


,battery_power,ram,internal_memory,primary_camera,front_camera,screen_size,mobile_weight,price_range
0,633,349,16,2,19,6.61,150,0
1,1661,2594,128,28,19,5.62,250,1
2,1482,4248,128,37,16,6.88,101,2
3,1158,7369,16,27,2,6.13,174,3
4,1149,5973,8,20,6,4.79,191,2


Number of records: 800

Price-range distribution:
price_range
0    200
1    200
2    200
3    200
Name: count, dtype: int64


In [2]:
# Separate input features and target
X = mobile.drop("price_range", axis=1)
y = mobile["price_range"]

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Train Random Forest classifier
model = RandomForestClassifier(
    n_estimators=150, random_state=42
)
model.fit(X_train, y_train)

# Predict and evaluate
y_pred = model.predict(X_test)

print("Actual price ranges:", y_test.to_numpy()[:10])
print("Predicted price ranges:", y_pred[:10])
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Actual price ranges: [2 1 1 3 2 1 2 0 0 3]
Predicted price ranges: [2 1 0 3 2 1 2 0 0 3]
Accuracy: 0.8375

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.82      0.85        40
           1       0.74      0.78      0.76        40
           2       0.82      0.82      0.82        40
           3       0.93      0.93      0.93        40

    accuracy                           0.84       160
   macro avg       0.84      0.84      0.84       160
weighted avg       0.84      0.84      0.84       160



In [3]:
# Predict the price range of a new mobile phone
new_mobile = pd.DataFrame([{
    "battery_power": 1800,
    "ram": 6000,
    "internal_memory": 128,
    "primary_camera": 32,
    "front_camera": 16,
    "screen_size": 6.5,
    "mobile_weight": 180
}])

predicted_range = model.predict(new_mobile)[0]
print("Predicted mobile price range:", predicted_range)
print("0 = Low, 1 = Medium, 2 = High, 3 = Very High")

Predicted mobile price range: 3
0 = Low, 1 = Medium, 2 = High, 3 = Very High


## Result

The Random Forest classifier was trained using mobile-phone specifications and used to predict four price ranges. Its performance was evaluated using accuracy and a classification report.